<a href="https://colab.research.google.com/github/dtrieu9/PDB-AI-Protein-Activity/blob/main/pdb_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Protein Classifier from the Protein Data Bank

This notebook downloads protein data from the RCSB Protein Data Bank, builds a simple dataset from functional keyword buckets, and trains a multiclass classifier.

We are looking at hemoglobin, lysozyme, insulin, and histone, but you can edit `CLASS_TERMS` to search for other proteins, too.

## 1) Imports and Helper Functions

This code cell contains many functions which will be used later in the notebook.

In [ ]:
!wget https://raw.githubusercontent.com/dtrieu9/PDB-AI-Protein-Activity/refs/heads/main/pdb_helpers.py

In [ ]:
import pdb_helpers as pdb

## 2) Configuration

This cell is for determining which website to use and what proteins to look for.

In [ ]:
SEARCH_URL = "https://search.rcsb.org/rcsbsearch/v2/query"
DATA_BASE = "https://data.rcsb.org/rest/v1/core"

# Edit these class buckets as needed.
CLASS_TERMS = {
    "hemoglobin" : "hemoglobin",
    "lysozyme" : "lysozyme",
    "insulin" : "insulin",
    "histone" : "histone"
}

AA_ORDER = list("ACDEFGHIKLMNPQRSTVWY")

## 3) Build the dataset from PDB

This may take a few minutes depending on `samples_per_class` and network speed.

In [ ]:
# Adjust these if you want a bigger or smaller dataset.
samples_per_class = 120
min_seq_len = 40

df = pdb.build_dataset(
    class_terms=CLASS_TERMS,
    samples_per_class=samples_per_class,
    min_seq_len=min_seq_len,
)

print("Dataset size:", len(df))
display(df.groupby("label").size().reset_index(name="count"))
display(df.head())

## 4) Visualization

We are using the `protein_visualization()` to randomly visualize one of the proteins we collected from the PDB. The next cell after that will show us amino acid chain sequences for each protein we collected.

In [ ]:
pdb.protein_visualization(df)

In [ ]:
with pdb.pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):
    formatted_output = [f"{index} {sequence}" for index, sequence in df['sequence'].items()]
    print('\n'.join(formatted_output))

## 5) Train and evaluate

There are many algorithms we could use to create a model that can classify proteins. Here, we apply our data to a Logistic Regression algorithm to create the model. Then we evaluate the model's performance.

In [ ]:
model = pdb.train_and_evaluate(df)

## 6) Save the trained model

In machine learning projects, training a model can be costly. You don't want to retrain the model every time you need to use it, so it's good practice to save it for later.

In [ ]:
model_path = "protein_classifier.joblib"
pdb.joblib.dump(
    {
        "model": model,
        "class_terms": CLASS_TERMS,
        "aa_order": AA_ORDER,
    },
    model_path,
)
print(f"Saved model to: {model_path}")

## 7) Predict on a new sequence

Let's see if the classifier can predict a protein you find yourself on the  PDB. Replace `example_sequence` with any protein sequence in one-letter amino-acid code.

1. Go to [rcsb.org](https://www.rcsb.org/)
2. Search for hemoglobin, lysozyme, insulin, or histone.
3. Click on one of the results
4. Download the FASTA sequence
5. Open the file in Notepad and copy one of the amino acid sequences.
6. Paste the sequence below where it says "example_sequence"



In [ ]:
example_sequence = "VLSPADKTNIKSTWDKIGGHAGDYGGEALDRTFQSFPTTKTYFPHFDLSPGSAQVKAHGKKVADALTTAVAHLDDLPGALSALSDLHAYKLRVDPVNFKLLSHCLLVTLACHHPTEFTPAVHASLDKFFAAVSTVLTSKYR"
print("Prediction:", model.predict([example_sequence])[0])
print("Probabilities:")
probabilities = dict(zip(model.classes_, model.predict_proba([example_sequence])[0]))
for protein, prob in probabilities.items():
    print(f"{protein}: {prob * 100:.2f}%")